In [1]:
# =========================
# LIMPIEZA DE ENTORNO
# =========================

!pip uninstall -y tensorflow tensorflow-cpu tensorflow-gpu protobuf transformers peft accelerate -q

# =========================
# INSTALACIÓN BASE (COMPATIBLE)
# =========================

!pip install transformers==4.38.2 accelerate==0.27.2 protobuf==3.20.3 -q

# =========================
# MÉTRICAS
# =========================

!pip install evaluate bert-score sacrebleu unbabel-comet bert-score -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 187.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 177.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires tensorflow>=2.2.0, which is not installed.
grain 0.2.17 requires protobuf>=5.28.3, but you have protobuf 3.20.3 which is incompatible.
grpc-google-iam-v1 0.14.4 requires protobuf<8.0.0,>=4.25.8, but you have protobuf 3.20.3 which is incompatible.
google-cloud-language 2.20.0 requires protobuf<8.0.0,>=4.25.8, but you 

In [1]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

path = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/train.csv"

df = pd.read_csv(path)
df = df[["source", "target"]].dropna()

Mounted at /content/drive


In [2]:
import re

def limpiar_texto(texto):
    return re.sub(r"[^\w\s-]", "", texto)

In [3]:
#Aplicamos limpieza

df["source_clean"] = df["source"].apply(limpiar_texto)
df["target_clean"] = df["target"].apply(limpiar_texto)

# SILABIFICADOR

In [4]:
VOWELS = {'a', 'e', 'i', 'o'}

DIGRAPHS = {'ch', 'sh', 'ts'}

CODAS = {'n', 's', 'sh', 'x'}

def tokenize(word):
    tokens = []
    i = 0

    while i < len(word):

        if i + 1 < len(word) and word[i:i+2] in DIGRAPHS:
            tokens.append(word[i:i+2])
            i += 2
        else:
            tokens.append(word[i])
            i += 1

    return tokens

def silabificador_shipibo(word):

    tokens = tokenize(word)

    syllables = []
    current = []

    i = 0

    while i < len(tokens):

        current.append(tokens[i])

        # si es vocal
        if tokens[i] in VOWELS:

            # mirar siguiente
            if i + 1 < len(tokens):

                nxt = tokens[i + 1]

                # caso VCV
                if nxt not in VOWELS:

                    # mirar siguiente del siguiente
                    if i + 2 < len(tokens):

                        nxt2 = tokens[i + 2]

                        # VCV → dividir antes de consonante
                        if nxt2 in VOWELS:
                            syllables.append(current)
                            current = []

                        # VCCV
                        else:

                            # coda válida
                            if nxt in CODAS:
                                current.append(nxt)
                                syllables.append(current)
                                current = []
                                i += 1
                            else:
                                syllables.append(current)
                                current = []

                    else:
                        # final
                        if nxt in CODAS:
                            current.append(nxt)
                            i += 1

            syllables.append(current)
            current = []

        i += 1

    if current:
        syllables.append(current)

    return [''.join(s) for s in syllables]

In [5]:
def silabificar_texto(texto):
    """
    El silabificador trabaja palabra por palabra.

    Cada palabra:
    palabra → sílabas

    Luego todas las sílabas se unen
    en una sola secuencia.
    """

    palabras = texto.split()

    resultado = []

    for palabra in palabras:

        silabas = silabificador_shipibo(palabra)

        resultado.extend(silabas)

    return resultado

In [6]:
# =========================
# EXTRAER SUBWORDS Y SÍLABAS
# =========================
from collections import Counter
subword_counter = Counter()
for texto in df["source_clean"]:
    palabras = texto.split()
    for palabra in palabras:
        silabas = silabificador_shipibo(palabra)
        subword_counter.update(silabas)

print("SUBWORDS MÁS FRECUENTES:")
print(subword_counter.most_common(50))

# ==============================================================================
# FILTRAR TOKENS ÚTILES DEL SILABIFICADOR (UMBRAL >= 50)
# ==============================================================================
nuevos_tokens = [token for token, count in subword_counter.items() if count >= 50]
print("Cantidad de nuevos tokens a agregar (freq >= 50):", len(nuevos_tokens))


SUBWORDS MÁS FRECUENTES:
[('', 283468), ('i', 49358), ('ja', 35675), ('a', 34753), ('ti', 19882), ('ka', 19741), ('ki', 19239), ('bo', 16370), ('ra', 16368), ('ma', 14632), ('na', 13157), ('bi', 12238), ('yo', 8828), ('ko', 8499), ('to', 8457), ('ke', 7977), ('we', 7876), ('in', 7871), ('o', 7640), ('xon', 7595), ('kin', 7232), ('ba', 7058), ('no', 6979), ('ri', 6827), ('ni', 6104), ('nan', 5949), ('an', 5824), ('jo', 5525), ('jas', 5192), ('ta', 5086), ('ya', 4656), ('tan', 4628), ('e', 4593), ('kon', 4572), ('shi', 4012), ('me', 3892), ('be', 3509), ('pa', 3377), ('non', 3343), ('kan', 3315), ('te', 3266), ('ton', 3131), ('ne', 3088), ('on', 3026), ('di', 2625), ('os', 2533), ('ká', 2506), ('pi', 2332), ('sha', 2286), ('mo', 2129)]
Cantidad de nuevos tokens a agregar (freq >= 50): 262


In [8]:
for i in range(5):

    print("ORIGINAL:", df.iloc[i]["source"])

    print("LIMPIO:", df.iloc[i]["source_clean"])


    print("TARGET:", df.iloc[i]["target_clean"])

    print("-----")

ORIGINAL: ¿matoki wishabo texea?
LIMPIO: matoki wishabo texea
TARGET: le sobran letras
-----
ORIGINAL: kenawe ainbo bakebo itan benbo bakebo chosko tsinki akanti
LIMPIO: kenawe ainbo bakebo itan benbo bakebo chosko tsinki akanti
TARGET: invita a las niñas y a los niños a formar grupos de cuatro
-----
ORIGINAL: moa wetsa baritian
LIMPIO: moa wetsa baritian
TARGET: en el año anterior
-----
ORIGINAL: shinanwe itan yoiwe jawekeska axonmein min jaska axon toponti shinanbo pikoti iki.
LIMPIO: shinanwe itan yoiwe jawekeska axonmein min jaska axon toponti shinanbo pikoti iki
TARGET: razona y argumenta generando ideas matemáticas
-----
ORIGINAL: shinan awe benbo bakebo itan ainbo bakebo betan jawekopiki min jaskara shinanbo akai ixon
LIMPIO: shinan awe benbo bakebo itan ainbo bakebo betan jawekopiki min jaskara shinanbo akai ixon
TARGET: establece con los niños y las niñas el propósito de los mensajes que van a escribir
-----


***Creación de dataset***

In [9]:
from datasets import Dataset
df_model = df[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
# Prepend prompt BEFORE dataset creation (mT5 specific)
df_model["source"] = df_model["source"].apply(
    lambda x: "translate Shipibo-Konibo to Spanish: " + x
)
dataset = Dataset.from_pandas(df_model)
print(dataset)


Dataset({
    features: ['source', 'target'],
    num_rows: 14728
})


In [ ]:
print(dataset[0])

***Tokenización***

In [10]:
## Tokenizer mT5

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

# ==============================================================================
# AGREGAR TOKENS AL TOKENIZER (Aquí se agregan los nuevos tokens al tokenizador)
# ==============================================================================
tokens_filtrados = [token for token in nuevos_tokens if token not in tokenizer.get_vocab()]
print("Nuevos tokens reales a agregar:", len(tokens_filtrados))
tokens_agregados = tokenizer.add_tokens(tokens_filtrados)
print("Tokens agregados con éxito:", tokens_agregados)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Nuevos tokens reales a agregar: 74
Tokens agregados con éxito: 73


In [11]:
# Prompt ya prependeado en la celda de creacion del dataset
print("Prompt ya configurado.")


Prompt ya configurado.


In [12]:
#Función de tokenización

def tokenizar_ejemplo(example):
    model_inputs = tokenizer(
        example["source"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        example["target"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [13]:
# Parche de compatibilidad: torchvision en Colab puede no tener VideoReader
try:
    import torchvision.io
    if not hasattr(torchvision.io, 'VideoReader'):
        class _DummyVideoReader:
            pass
        torchvision.io.VideoReader = _DummyVideoReader
except ImportError:
    pass

#Aplicación de tokenización

tokenized_dataset = dataset.map(tokenizar_ejemplo)
tokenized_dataset = tokenized_dataset.remove_columns(["source", "target"])
tokenized_dataset.set_format("torch")

Map:   0%|          | 0/14728 [00:00<?, ? examples/s]

In [14]:
# ==============================================================================
# DIAGNÓSTICO DE TOKENIZACIÓN (VERIFICACIÓN DE ESTRUCTURA Y EOS)
# ==============================================================================
ejemplo_idx = 0
input_ids = tokenized_dataset[ejemplo_idx]["input_ids"]
label_ids = tokenized_dataset[ejemplo_idx]["labels"]

print("INPUT IDS:", input_ids)
print("INPUT DECODED:", tokenizer.decode(input_ids))
print("---")
print("LABEL IDS (con -100 reemplazado por pad para decodificar):", label_ids.tolist() if hasattr(label_ids, 'tolist') else label_ids)
# Reemplazar -100 por pad_token_id (0) para decodificar
labels_list = label_ids.tolist() if hasattr(label_ids, 'tolist') else label_ids
labels_decodificables = [(l if l != -100 else tokenizer.pad_token_id) for l in labels_list]
print("LABEL DECODED:", tokenizer.decode(labels_decodificables))


INPUT IDS: tensor([ 37194,  63211,  40688,    264,  29187,  40688,    288,    259,  29037,
           267,    259, 187566,    266,  22491,  15973,    400,   9878,    262,
             1,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0])
INPUT DECODED: translate Shipibo-Konibo to Spanish: matoki wishabo texea</s><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad><pad>
---
LABEL IDS (con -100 reemplazado por pad para decodificar): [340, 259, 250149, 29148, 340, 35570, 1, 0, 0, 

# Modelo

In [15]:
##Traer el modelo

from transformers import AutoModelForSeq2SeqLM

model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))

model.to("cuda")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


pytorch_model.bin:   0%|          | 0.00/1.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

MT5ForConditionalGeneration(
  (shared): Embedding(250173, 512)
  (encoder): MT5Stack(
    (embed_tokens): Embedding(250173, 512)
    (block): ModuleList(
      (0): MT5Block(
        (layer): ModuleList(
          (0): MT5LayerSelfAttention(
            (SelfAttention): MT5Attention(
              (q): Linear(in_features=512, out_features=384, bias=False)
              (k): Linear(in_features=512, out_features=384, bias=False)
              (v): Linear(in_features=512, out_features=384, bias=False)
              (o): Linear(in_features=384, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 6)
            )
            (layer_norm): MT5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): MT5LayerFF(
            (DenseReluDense): MT5DenseGatedActDense(
              (wi_0): Linear(in_features=512, out_features=1024, bias=False)
              (wi_1): Linear(in_features=512, out_features=1024, bias=False)
          

In [16]:
##Parámetros para entrenamiento

from transformers import TrainingArguments, Trainer

# ==============================================================================
# PARÁMETROS FASE 1 (ENTRENAMIENTO INICIAL - OPTIMIZADO PARA AHORRO DE TIEMPO):
# - per_device_train_batch_size=16: Lote grande de 16 para acelerar drásticamente el entrenamiento.
# - num_train_epochs=6: Reducido de 10 a 6 para reducir consumo de créditos de Colab.
# - learning_rate=6e-4: Tasa alta proporcional al tamaño de lote grande (16).
# - warmup_steps=150: Pasos de calentamiento adaptados para el lote.
# - weight_decay=0.01: Regularización estándar.
# - logging_steps=150: Frecuencia de log de pérdidas adaptada.
# ==============================================================================
training_args = TrainingArguments(
    output_dir="./results_mt5",
    per_device_train_batch_size=16,
    num_train_epochs=6,
    learning_rate=6e-4,
    warmup_steps=150,
    weight_decay=0.01,
    logging_steps=150,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [17]:
##Entrenar el modelo

trainer.train()

Step,Training Loss
150,27.294400
300,2.750100
450,1.747700
600,1.516900
750,1.432400
900,1.370000
1050,1.323000
1200,1.281000
1350,1.259400
1500,1.233600


TrainOutput(global_step=5526, training_loss=1.875229909029121, metrics={'train_runtime': 436.3744, 'train_samples_per_second': 202.505, 'train_steps_per_second': 12.663, 'total_flos': 5841628890660864.0, 'train_loss': 1.875229909029121, 'epoch': 6.0})

In [18]:
#Guardar el modelo

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4/spiece.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4/tokenizer.json')

In [19]:
# ==============================================================================
# CARGA AUTOMÁTICA DEL MODELO ENTRENADO Y TOKENIZER (EVITA ENTRENAR DESDE CERO)
# ==============================================================================
import os
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando modelo entrenado v4 y tokenizer desde: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_v4, local_files_only=True)
    model.to("cuda")
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando modelo entrenado previo y tokenizer desde: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_orig, local_files_only=True)
    model.to("cuda")
else:
    print("[ADVERTENCIA] No se encontró ningún checkpoint en Google Drive.")
    print("[INFO] Se continuará usando el modelo base en memoria.")


[INFO] Cargando modelo entrenado v4 y tokenizer desde: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4


You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers
/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [20]:
def traducir(texto):
    texto_con_prompt = "translate Shipibo-Konibo to Spanish: " + texto
    inputs = tokenizer(texto_con_prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_length=50,
        num_beams=4,           #  beam search
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# VALIDACIÓN (val.csv)

In [21]:
#Cargar val

path_val = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/val.csv"

df_val = pd.read_csv(path_val)
df_val = df_val[["source", "target"]].dropna()

In [22]:
#Preprocesamiento de val
df_val["source_clean"] = df_val["source"].apply(limpiar_texto)
df_val["target_clean"] = df_val["target"].apply(limpiar_texto)

from datasets import Dataset
df_val_model = df_val[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
# Prepend prompt BEFORE dataset creation (mT5 specific)
df_val_model["source"] = df_val_model["source"].apply(
    lambda x: "translate Shipibo-Konibo to Spanish: " + x
)
val_dataset = Dataset.from_pandas(df_val_model)
print("Dataset de validación cargado con éxito:", val_dataset)


Dataset de validación cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 1841
})


In [23]:
#Ejemplos de los resultados

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", df_val.iloc[i]["target_clean"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")

INPUT: jatianra ja dios meniti jawéki min boá jainxon dios yoina menoxontiainkobipari abaini ja joni betan joi benxoax raeananipari mia kati jake jatian moa jaskáa pekáoparikayara min ati jake dios jawéki menikin
REAL: deja allí mismo tu ofrenda ante el altar y vete antes a hacer las paces con tu hermano después vuelve y presenta tu ofrenda
PRED: y cuando vengas a la iglesia y a la iglesia y a la iglesia y a la iglesia y a la iglesia y a
-----
INPUT: jain yoyo ati kirika benxoati
REAL: biblioteca del aula
PRED: libros de biblioteca
-----
INPUT: teayamawe yoyo ikashamaitian
REAL: no lo obligues
PRED: levántate y levántate
-----
INPUT: moara jaton aká jawékibo jan jato yoixonti nete senenke
REAL: adoren al que hizo el cielo la tierra el mar y los manantiales de agua
PRED: y cuando llegue el día de hoy
-----
INPUT: rotacion itan traslacion
REAL: rotación y traslación
PRED: rotación y traslación
-----


# Métricas para val.csv

In [24]:
#Preparar listas

preds = []
refs = []

for i in range(len(df_val)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = df_val.iloc[i]["target_clean"]

    preds.append(pred)
    refs.append([ref])

In [25]:
#BLEU

import evaluate

bleu = evaluate.load("bleu")
print("BLEU:", bleu.compute(predictions=preds, references=refs))

BLEU: {'bleu': 0.04020589810835257, 'precisions': [0.2177096636866752, 0.058709409210587093, 0.023674911660777384, 0.011942454035438537], 'brevity_penalty': 0.9221422636008344, 'length_ratio': 0.9250216586595259, 'translation_length': 23490, 'reference_length': 25394}


In [26]:
#CHRF

chrf = evaluate.load("chrf")
print("ChrF:", chrf.compute(predictions=preds, references=refs))

ChrF: {'score': 21.077981180268736, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [27]:
#COMET

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt20-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(df_val)):
    data.append({
        "src": df_val.iloc[i]["source"],  # ORIGINAL
        "mt": preds[i],
        "ref": df_val.iloc[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8, gpus=1)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
INFO:pytorch_lightning.utilities.rank_zero:You are using a CUDA device ('NVIDIA RTX PRO 6000 Blackwell Server Edition') th

COMET: -1.2179130992110028


In [28]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


METEOR: {'meteor': 0.16587390741652003}


In [29]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

BERTScore Precision (Promedio): 0.7161961690619353
BERTScore Recall (Promedio): 0.7030202426837877
BERTScore F1 (Promedio): 0.7090854057146764


In [30]:
## REENTRENAR EL MODELO, PARA REVISION DE MEJORA
# ==============================================================================
# CAMBIOS PARA EL REENTRENAMIENTO INCREMENTAL (FASE 2 - OPTIMIZADO):
# 1. learning_rate=9e-5: Ajuste fino más suave y controlado.
# 2. num_train_epochs=3: Reducido para minimizar tiempo de ejecución en Colab.
# 3. per_device_train_batch_size=16: Lote de 16.
# 4. warmup_steps=75: Calentamiento de gradiente corto.
# ==============================================================================

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=16,
    num_train_epochs=3,
    learning_rate=9e-5,
    warmup_steps=75,
    weight_decay=0.01,
    logging_steps=150,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    report_to="none"
)

In [31]:
##Limpiamos el entorno

import torch
torch.cuda.empty_cache()

In [32]:
import torch
import gc

del model
del trainer

gc.collect()
torch.cuda.empty_cache()

In [33]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import os

# ==============================================================================
# REENTRENAMIENTO INCREMENTAL RESILIENTE:
# Busca los pesos de la Fase 1 en Google Drive (v4 o original) para continuar el aprendizaje.
# Si no encuentra ninguno, cae al modelo base de Hugging Face por seguridad.
# ==============================================================================
path_v4 = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4"
path_orig = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v1"

if os.path.exists(path_v4):
    print(f"[INFO] Cargando pesos de Fase 1 (v4) y tokenizer para reentrenamiento: {path_v4}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_v4, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_v4, local_files_only=True)
elif os.path.exists(path_orig):
    print(f"[INFO] Cargando pesos de Fase 1 (original) y tokenizer para reentrenamiento: {path_orig}")
    model = AutoModelForSeq2SeqLM.from_pretrained(path_orig, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(path_orig, local_files_only=True)
else:
    print("[ADVERTENCIA] No se encontró modelo previo. Cargando modelo base desde cero...")
    model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-small")

model.to("cuda")
# =========================
# REDIMENSIONAR EMBEDDINGS
# =========================
model.resize_token_embeddings(len(tokenizer))


[INFO] Cargando pesos de Fase 1 (v4) y tokenizer para reentrenamiento: /content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:550: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Embedding(250173, 512)

In [34]:
##Creamos del trainer despues de limpiar memoria

from transformers import Trainer
from transformers import TrainingArguments

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset
)

In [35]:
#Realizamos el entrenamiento despues de los siguientes cambios de parametros

trainer.train()

Step,Training Loss
150,0.953100
300,0.955900
450,0.910100
600,0.896000
750,0.885200
900,0.879900
1050,0.877600
1200,0.869200
1350,0.863700
1500,0.860800


TrainOutput(global_step=2763, training_loss=0.867947210145523, metrics={'train_runtime': 217.1509, 'train_samples_per_second': 203.471, 'train_steps_per_second': 12.724, 'total_flos': 2920814445330432.0, 'train_loss': 0.867947210145523, 'epoch': 3.0})

In [36]:
##Guardar 2da version

trainer.save_model("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4_retrained")
tokenizer.save_pretrained("/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4_retrained")

('/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4_retrained/tokenizer_config.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4_retrained/special_tokens_map.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4_retrained/spiece.model',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4_retrained/added_tokens.json',
 '/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/modelo_mT5_ShipiboKonibo_v4_retrained/tokenizer.json')

In [37]:
# Carga y preparación del dataset de validación para evaluar el modelo después del reentrenamiento
print("Generando predicciones sobre el conjunto de validación con el modelo reentrenado...")

#Probar ejemplos

for i in range(5):
    print("INPUT:", df_val.iloc[i]["source_clean"])
    print("REAL:", val_dataset[i]["target"])
    print("PRED:", traducir(df_val.iloc[i]["source_clean"]))
    print("-----")


Generando predicciones sobre el conjunto de validación con el modelo reentrenado...
INPUT: jatianra ja dios meniti jawéki min boá jainxon dios yoina menoxontiainkobipari abaini ja joni betan joi benxoax raeananipari mia kati jake jatian moa jaskáa pekáoparikayara min ati jake dios jawéki menikin
REAL: deja allí mismo tu ofrenda ante el altar y vete antes a hacer las paces con tu hermano después vuelve y presenta tu ofrenda
PRED: y si tú eres el sa lvador de dios y de cristo eres el sa lvador de la palabra de dios
-----
INPUT: jain yoyo ati kirika benxoati
REAL: biblioteca del aula
PRED: lee la biblioteca
-----
INPUT: teayamawe yoyo ikashamaitian
REAL: no lo obligues
PRED: cuál es la lectura
-----
INPUT: moara jaton aká jawékibo jan jato yoixonti nete senenke
REAL: adoren al que hizo el cielo la tierra el mar y los manantiales de agua
PRED: felicítalos todas las cosa s
-----
INPUT: rotacion itan traslacion
REAL: rotación y traslación
PRED: rotador y traslación
-----


In [38]:
#Generar predicciones

preds = []
refs = []

for i in range(len(val_dataset)):
    pred = traducir(df_val.iloc[i]["source_clean"])
    ref = val_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [39]:
##Aplicamos nuevamente las métricas


##BLEU

import evaluate

bleu = evaluate.load("bleu")
bleu.compute(predictions=preds, references=refs)


{'bleu': 0.03200869449338024,
 'precisions': [0.21668391637342166,
  0.04969974007349646,
  0.01680015628052354,
  0.0071233463660221735],
 'brevity_penalty': 0.9499995880619696,
 'length_ratio': 0.9512089469953532,
 'translation_length': 24155,
 'reference_length': 25394}

In [40]:
##CHRF

chrf = evaluate.load("chrf")

chrf_result = chrf.compute(predictions=preds, references=refs)

print("ChrF:", chrf_result)

ChrF: {'score': 21.29523683181203, 'char_order': 6, 'word_order': 0, 'beta': 2}


In [41]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(val_dataset)):
    data.append({
        "src": df_val.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": val_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

LICENSE: 0.00B [00:00, ?B/s]

hparams.yaml:   0%|          | 0.00/567 [00:00<?, ?B/s]

checkpoints/model.ckpt:   0%|          | 0.00/2.32G [00:00<?, ?B/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: Fals

COMET: 0.47401208927099114


In [42]:
# =========================
# MÉTRICA METEOR (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR (Validación - Reentrenado):", meteor_result)
except Exception as e:
    print("Error al calcular METEOR en validación con reentrenamiento:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR (Validación - Reentrenado): {'meteor': 0.1541669019360507}


In [43]:
# =========================
# MÉTRICA BERTSCORE (VALIDACIÓN - REENTRENADO)
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Validación - Reentrenado - Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Validación - Reentrenado - Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Validación - Reentrenado - Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore en validación con reentrenamiento:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BERTScore Precision (Validación - Reentrenado - Promedio): 0.710244565538451
BERTScore Recall (Validación - Reentrenado - Promedio): 0.7028946706550656
BERTScore F1 (Validación - Reentrenado - Promedio): 0.7061471259833548


In [44]:
##PROBAMOS EL TEST.CSV

path_test = "/content/drive/MyDrive/Tesis 2/Entrenamiento modelos NMT/Shipibo Konibo/test.csv"

df_test = pd.read_csv(path_test)
df_test = df_test[["source", "target"]].dropna()

In [45]:
#Preprocesamiento de test
df_test["source_clean"] = df_test["source"].apply(limpiar_texto)
df_test["target_clean"] = df_test["target"].apply(limpiar_texto)


In [46]:
from datasets import Dataset

# ==============================================================================
# CREACIÓN DEL DATASET DE PRUEBA:
# Corregimos la NameError asegurando que se carga a partir de df_test (y no del df de entrenamiento).
# ==============================================================================
df_test_model = df_test[["source_clean", "target_clean"]].rename(
    columns={
        "source_clean": "source",
        "target_clean": "target"
    }
)
test_dataset = Dataset.from_pandas(df_test_model)
print("Dataset de prueba cargado con éxito:", test_dataset)


Dataset de prueba cargado con éxito: Dataset({
    features: ['source', 'target'],
    num_rows: 1841
})


In [47]:
preds = []
refs = []

for i in range(len(test_dataset)):
    pred = traducir(df_test.iloc[i]["source_clean"])
    ref = test_dataset[i]["target"]

    preds.append(pred)
    refs.append([ref])

In [48]:
##BLEU

bleu.compute(predictions=preds, references=refs)

{'bleu': 0.03405668379869197,
 'precisions': [0.21498371335504887,
  0.05216014476362814,
  0.018306523240896082,
  0.007315883596163225],
 'brevity_penalty': 0.9728549782419728,
 'length_ratio': 0.9732168258484047,
 'translation_length': 23946,
 'reference_length': 24605}

In [ ]:
##CHRF
chrf.compute(predictions=preds, references=refs)

In [49]:
##COMET - Evaluación

from comet import download_model, load_from_checkpoint

model_path = download_model("Unbabel/wmt22-comet-da")
comet_model = load_from_checkpoint(model_path)

data = []

for i in range(len(test_dataset)):
    data.append({
        "src": df_test.iloc[i]["source_clean"],
        "mt": preds[i],
        "ref": test_dataset[i]["target"]
    })

scores = comet_model.predict(data, batch_size=8)

print("COMET:", scores["system_score"])

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.5. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: Fals

COMET: 0.47364972366676716


In [50]:
# =========================
# MÉTRICA METEOR
# =========================
try:
    import evaluate
    meteor = evaluate.load("meteor")
    meteor_result = meteor.compute(predictions=preds, references=[r[0] for r in refs])
    print("METEOR:", meteor_result)
except Exception as e:
    print("Error al calcular METEOR:", e)


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


METEOR: {'meteor': 0.1508414669664913}


In [51]:
# =========================
# MÉTRICA BERTSCORE
# =========================
try:
    import evaluate
    bertscore = evaluate.load("bertscore")
    bertscore_result = bertscore.compute(predictions=preds, references=[r[0] for r in refs], lang="es")
    print("BERTScore Precision (Promedio):", sum(bertscore_result["precision"]) / len(bertscore_result["precision"]))
    print("BERTScore Recall (Promedio):", sum(bertscore_result["recall"]) / len(bertscore_result["recall"]))
    print("BERTScore F1 (Promedio):", sum(bertscore_result["f1"]) / len(bertscore_result["f1"]))
except Exception as e:
    print("Error al calcular BERTScore:", e)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


BERTScore Precision (Promedio): 0.7083508913151795
BERTScore Recall (Promedio): 0.7024408320186579
BERTScore F1 (Promedio): 0.7049731003370445
